# Notebook 01 â€” Stereo Depth (RAFT-Stereo)

Downloads SceneFlow, trains RAFT-Stereo on indoor data, exports ONNX for Jetson TensorRT.

## Cell 01-01 | Mount Drive & Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, torch

BASE   = '/content/drive/MyDrive/ARGUS'
DS     = f'{BASE}/datasets'
MODELS = f'{BASE}/models'
CKPTS  = f'{BASE}/checkpoints'
EXPORTS = f'{BASE}/exports'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | Drive: {os.path.exists(BASE)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda | Drive: True


## Cell 01-02 | Clone RAFT-Stereo Repository

In [6]:
RAFT_DIR = f'{BASE}/raft_stereo'

if not os.path.exists(RAFT_DIR):
    os.system(f'git clone https://github.com/princeton-vl/RAFT-Stereo {RAFT_DIR}')
    print('RAFT-Stereo cloned to Drive')
else:
    print('RAFT-Stereo already exists')

os.chdir(RAFT_DIR)
os.system('pip install -q einops')
print('RAFT-Stereo directory:')
os.system('ls')

RAFT-Stereo already exists
RAFT-Stereo directory:


0

## Cell 01-03 | Download SceneFlow Dataset

In [5]:
import os

# SceneFlow (FlyingThings3D) is the pretraining dataset for RAFT-Stereo.
# The pretrained weights already encode SceneFlow knowledge — you do NOT
# need to re-download it. What you DO need for fine-tuning is your own
# AR0234 stereo pairs collected on the Jetson.
#
# Expected directory layout for fine-tuning:
#   {DS}/custom_stereo/train/left/        ← left camera PNG frames
#   {DS}/custom_stereo/train/right/       ← matching right camera frames
#   {DS}/custom_stereo/train/disparity/   ← disparity maps (uint16 PNG, px * 256)
#
# How to collect:
#   1. Run scripts/collect_stereo_pairs.py on the Jetson while walking around
#   2. scp the frames to Google Drive under the paths above
#   3. Re-run this notebook — the training cell will auto-detect them

CUSTOM_STEREO = f'{DS}/custom_stereo'
for split in ['train', 'val']:
    for sub in ['left', 'right', 'disparity']:
        os.makedirs(f'{CUSTOM_STEREO}/{split}/{sub}', exist_ok=True)

n_left = len([f for f in os.listdir(f'{CUSTOM_STEREO}/train/left') if f.endswith('.png')])
if n_left > 0:
    print(f'Custom stereo pairs ready: {n_left} training frames found')
else:
    print('No custom stereo pairs yet — RAFT-Stereo will use pretrained weights as-is.')
    print(f'Add AR0234 stereo frames to: {CUSTOM_STEREO}/train/left|right|disparity')


SceneFlow already downloaded


0

## Cell 01-04 | Download Pretrained RAFT-Stereo Weights

In [7]:
# Download RAFT-Stereo pretrained weights via the repo's own download script.
# The authors host weights on Google Drive; download_models.sh uses gdown.
# gdown --continue resumes interrupted downloads automatically.
import subprocess, os

WEIGHTS_DIR = f'{MODELS}/depth'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
CKPT_PATH = f'{WEIGHTS_DIR}/raftstereo-realtime.pth'
MIN_MB    = 20

def _ckpt_ok(p): return os.path.exists(p) and os.path.getsize(p)/1e6 >= MIN_MB

if _ckpt_ok(CKPT_PATH):
    print(f"Checkpoint ready ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)")
else:
    if os.path.exists(CKPT_PATH):
        print("Partial checkpoint — removing"); os.remove(CKPT_PATH)
    subprocess.run(["pip", "install", "-q", "gdown"], check=True)
    os.chdir(RAFT_DIR)
    ret = subprocess.run(["bash", "download_models.sh"],
                         capture_output=True, text=True)
    print(ret.stdout[-2000:] or "(no stdout)")
    if ret.returncode != 0:
        print("stderr:", ret.stderr[-500:])
    repo_models = f"{RAFT_DIR}/models"
    if os.path.isdir(repo_models):
        for fn in os.listdir(repo_models):
            if fn.endswith(".pth"):
                os.replace(f"{repo_models}/{fn}", f"{WEIGHTS_DIR}/{fn}")
                print(f"Moved {fn} -> {WEIGHTS_DIR}/")

if _ckpt_ok(CKPT_PATH):
    print(f"Checkpoint ready ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)")
else:
    raise RuntimeError("Checkpoint download failed — check gdown output above")


HTTPError: HTTP Error 404: Not Found

## Cell 01-05 | Custom Indoor Dataset Loader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

class StereoDataset(Dataset):
    # Custom stereo dataset for AR0234 indoor image pairs
    def __init__(self, root, split='train', img_size=(480, 640)):
        self.left_dir  = f'{root}/{split}/left'
        self.right_dir = f'{root}/{split}/right'
        self.disp_dir  = f'{root}/{split}/disp'
        self.img_size  = img_size
        self.samples   = sorted([f for f in os.listdir(self.left_dir) if f.endswith(('.png', '.jpg'))])
        print(f'  {split}: {len(self.samples)} stereo pairs found')
        self.transform = A.Compose([
            A.RandomCrop(img_size[0], img_size[1]),
            A.ColorJitter(brightness=0.2, contrast=0.2, p=0.5),
            A.ToFloat(max_value=255),
            ToTensorV2(),
        ], additional_targets={'right': 'image'})

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        name  = self.samples[idx]
        stem  = os.path.splitext(name)[0]
        left  = np.array(Image.open(f'{self.left_dir}/{name}').convert('RGB'))
        right = np.array(Image.open(f'{self.right_dir}/{name}').convert('RGB'))
        disp  = np.load(f'{self.disp_dir}/{stem}.npy').astype(np.float32)
        out   = self.transform(image=left, right=right)
        disp_t = torch.from_numpy(disp[:self.img_size[0], :self.img_size[1]]).unsqueeze(0)
        return {'left': out['image'], 'right': out['right'], 'disparity': disp_t}

CUSTOM_STEREO = f'{DS}/custom_stereo'
for d in [f'{CUSTOM_STEREO}/train/left', f'{CUSTOM_STEREO}/train/right', f'{CUSTOM_STEREO}/train/disp']:
    os.makedirs(d, exist_ok=True)
print('Custom stereo dataset structure ready.')
print('Add AR0234 stereo pairs before running the training cell.')

## Cell 01-06 | RAFT-Stereo Training Loop

In [ ]:
import sys, time, shutil
sys.path.insert(0, RAFT_DIR)
from core.raft_stereo import RAFTStereo
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import torch.nn.functional as F
from tqdm import tqdm

args = type('Args', (), {
    'hidden_dims': [128]*3, 'context_dims': [128]*3, 'context_norm': 'batch',
    'corr_implementation': 'reg', 'shared_backbone': False,
    'corr_levels': 4, 'corr_radius': 4, 'n_downsample': 2,
    'slow_fast_gru': True, 'n_gru_layers': 3, 'mixed_precision': True,
})()

CUSTOM_STEREO = f'{DS}/custom_stereo'
PRETRAINED    = f'{MODELS}/depth/raftstereo-realtime.pth'
FINAL         = f'{MODELS}/depth/raft_stereo_final.pth'
CKPT_DIR      = f'{CKPTS}/raft_stereo_indoor'
LATEST        = f'{CKPT_DIR}/latest.pth'
os.makedirs(CKPT_DIR, exist_ok=True)

dataset = StereoDataset(CUSTOM_STEREO, 'train', img_size=(320, 480))
n_pairs = len(dataset)

if n_pairs == 0:
    print('No custom stereo pairs found — using pretrained RAFT-Stereo directly.')
    print('  To fine-tune later, add matched left/right/disparity PNGs to:')
    print(f'  {CUSTOM_STEREO}/train/left|right|disparity')
    model = RAFTStereo(args).to(DEVICE)
    ckpt  = torch.load(PRETRAINED, map_location=DEVICE)
    model.load_state_dict(ckpt, strict=False)
    torch.save(model.state_dict(), FINAL)
    print(f'Pretrained weights saved as final model: {FINAL}')
else:
    print(f'{n_pairs} custom stereo pairs found — starting fine-tuning...')
    EPOCHS = 20
    loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=8,
                        pin_memory=True, drop_last=True)

    model     = RAFTStereo(args).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = OneCycleLR(optimizer, max_lr=1e-4,
                           steps_per_epoch=len(loader), epochs=EPOCHS, pct_start=0.05)
    scaler    = torch.cuda.amp.GradScaler()

    start_epoch = 0
    if os.path.exists(LATEST):
        try:
            state = torch.load(LATEST, map_location=DEVICE)
            model.load_state_dict(state['model_state'])
            optimizer.load_state_dict(state['optimizer_state'])
            scheduler.load_state_dict(state['scheduler_state'])
            scaler.load_state_dict(state['scaler_state'])
            start_epoch = state['epoch']
            print(f'Resumed from epoch {start_epoch}/{EPOCHS}  ({LATEST})')
        except Exception as e:
            print(f'Checkpoint corrupt ({e}) — removing and starting from scratch')
            os.remove(LATEST)
            start_epoch = 0
            model     = RAFTStereo(args).to(DEVICE)
            optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
            scheduler = OneCycleLR(optimizer, max_lr=1e-4,
                                   steps_per_epoch=len(loader), epochs=EPOCHS, pct_start=0.05)
            scaler    = torch.cuda.amp.GradScaler()

    if start_epoch == 0:
        ckpt = torch.load(PRETRAINED, map_location=DEVICE)
        model.load_state_dict(ckpt, strict=False)
        print('Pretrained weights loaded — starting from epoch 0')

    if start_epoch >= EPOCHS:
        print(f'Fine-tuning already complete ({EPOCHS} epochs).')
    else:
        t_train_start = time.time()
        model.train()
        epoch_bar = tqdm(range(start_epoch, EPOCHS), desc='RAFT Training', unit='epoch')
        for epoch in epoch_bar:
            epoch_loss = 0.0
            t_epoch    = time.time()
            batch_bar  = tqdm(loader, desc=f'Ep {epoch+1}/{EPOCHS}', unit='batch',
                              leave=False, dynamic_ncols=True)
            for batch in batch_bar:
                left  = batch['left'].to(DEVICE)
                right = batch['right'].to(DEVICE)
                disp  = batch['disparity'].to(DEVICE)
                optimizer.zero_grad()
                with torch.cuda.amp.autocast():
                    preds = model(left, right, iters=12)
                    loss  = 0.0
                    for j, pred in enumerate(preds):
                        w    = 0.9 ** (len(preds) - j - 1)
                        mask = (disp > 0) & (disp < 192)
                        loss += w * F.smooth_l1_loss(pred[mask], disp[mask])
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update(); scheduler.step()
                epoch_loss += loss.item()
                batch_bar.set_postfix(loss=f'{loss.item():.4f}',
                                      lr=f'{scheduler.get_last_lr()[0]:.1e}')

            avg_loss  = epoch_loss / max(len(loader), 1)
            elapsed   = time.time() - t_train_start
            ep_time   = time.time() - t_epoch
            remaining = ep_time * (EPOCHS - epoch - 1)
            epoch_bar.set_postfix(avg_loss=f'{avg_loss:.4f}',
                                   ep=f'{ep_time/60:.1f}m',
                                   elapsed=f'{elapsed/3600:.1f}h',
                                   eta=f'{remaining/3600:.1f}h')
            print(f'Epoch {epoch+1}/{EPOCHS} | loss={avg_loss:.4f} | '
                  f'ep={ep_time/60:.1f}m | elapsed={elapsed/3600:.2f}h | eta={remaining/3600:.1f}h')

            torch.save({
                'epoch'          : epoch + 1,
                'model_state'    : model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state'   : scaler.state_dict(),
                'avg_loss'       : avg_loss,
            }, LATEST)
            if (epoch + 1) % 5 == 0:
                numbered = f'{CKPT_DIR}/epoch_{epoch+1}.pth'
                shutil.copy(LATEST, numbered)
                print(f'  Checkpoint: {numbered}')

        total_h = (time.time() - t_train_start) / 3600
        torch.save(model.state_dict(), FINAL)
        print(f'Fine-tuning complete in {total_h:.2f}h. Final model: {FINAL}')


## Cell 01-07 | Export to ONNX

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxscript'])

model.eval()
H, W       = 480, 640
dummy_left  = torch.randn(1, 3, H, W).to(DEVICE)
dummy_right = torch.randn(1, 3, H, W).to(DEVICE)
ONNX_PATH   = f"{EXPORTS}/tensorrt/raft_stereo_640x480.onnx"
os.makedirs(os.path.dirname(ONNX_PATH), exist_ok=True)

class RAFTStereoExport(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.model = m
    def forward(self, l, r): return self.model(l, r, iters=12, test_mode=True)

# Use legacy TorchScript exporter (dynamo=False) — required for RAFT-Stereo
# because cudnn_grid_sampler is not supported by the new dynamo exporter.
with torch.no_grad():
    torch.onnx.export(
        RAFTStereoExport(model), (dummy_left, dummy_right), ONNX_PATH,
        dynamo=False,
        input_names=["left", "right"], output_names=["disparity"],
        dynamic_axes={"left": {0: "batch"}, "right": {0: "batch"}, "disparity": {0: "batch"}},
        opset_version=16, do_constant_folding=True,
    )
print(f"ONNX exported: {ONNX_PATH}")
print(f"Size: {os.path.getsize(ONNX_PATH)/1e6:.1f} MB")
print("On Jetson: trtexec --onnx=raft_stereo_640x480.onnx --saveEngine=raft_stereo_fp16.trt --fp16")
